# 🔬 Molecular Signature Preprocessing Pipeline (Signaturizer)

**Objective**  
This notebook implements a reproducible preprocessing pipeline for generating
**bioactivity-informed molecular embeddings** using the **Signaturizer framework**,
designed for large-scale downstream machine learning tasks in computational biology.

**Scientific Context**
- Molecular signatures capture pathway-level biological activity beyond structural similarity
- Enables mechanism-aware modeling of complex phenotypes (e.g., aging, stress response, metabolism)
- Designed for scalability, interpretability, and downstream ML compatibility

**Design principles**  
Reproducible • Modular • Dataset-agnostic • Safe checks (assertions) • Clean outputs

**Key Outputs**
- Cleaned molecular input set
- Signaturizer embeddings (A–E spaces)
- ML-ready feature matrices with metadata alignment

> 📌 *This notebook is intended as a reusable preprocessing module and forms the foundation for multiple predictive pipelines.*

> 💡*Tip: If you're reviewing this quickly, search for **“Feature_Signaturizer”** — that's the core function.*

# Importing Libraries

In [ ]:
# Housekeeping: imports + display
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"   # must be set BEFORE importing tensorflow/signaturizer

In [ ]:
!pip install -U "tf-keras~=2.16" tensorflow-hub ## restart the kernel session once after running this cell

In [ ]:
#importing libraries
import sys

import numpy as np
import pandas as pd
from pathlib import Path

# Optional: prettier DataFrame display
pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 120)

from collections import Counter

import warnings
warnings.filterwarnings("ignore")

import pickle
import joblib
import sklearn
from sklearn.feature_selection import VarianceThreshold

# 🧬 Molecular Signature Generation

**Rationale**  
Signaturizer embeddings encode inferred biological activity patterns derived from
large-scale perturbation data, enabling downstream models to operate in a
mechanistically informed feature space.

**Embedding Spaces**
- **A–E spaces** capture complementary biological resolutions
- Output used directly for ML or similarity analysis




In [ ]:
!pip install rdkit

import rdkit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.4/36.4 MB 54.6 MB/s eta 0:00:00


In [ ]:
!pip install pytest
!pip install pytest-cov
!pip install tqdm

import pytest
import tensorflow as tf
import tensorflow_hub as hub
from tqdm import tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.5/253.5 kB 19.3 MB/s eta 0:00:00


In [ ]:
!pip install signaturizer

from signaturizer import Signaturizer

INFO: pip is looking at multiple versions of signaturizer to determine which version is compatible with other requirements. This could take a while.


# 🧭 Pipeline Overview

**Input → Processing → Output**

1. **Input ingestion**
   - SMILES / compound identifiers
   - Metadata harmonization

2. **Standardization**
   - Canonicalization
   - Deduplication
   - Error filtering

3. **Signature generation**
   - Signaturizer model loading
   - Bioactivity embedding extraction (A–E spaces)

4. **Post-processing**
   - Dimensional checks
   - Missing value handling
   - Export for ML consumption

**Design Principles**
- Deterministic execution
- Modular logic
- Dataset-agnostic


## ⚙️ Environment & Reproducibility

**Core dependencies**
- Python ≥ 3.9
- Signaturizer
- RDKit
- NumPy / Pandas

**Execution and Reproducibility Notes**
- Designed to run on CPU (GPU optional)
- Memory-safe for large compound libraries
- Set seeds where randomness exists (if applicable).
- Keep this notebook lightweight: move heavy helpers to `utils.py` when productizing.
- Prefer saving artifacts to `data/processed/` and `features/`.

> 🧪 *Tested on Linux and macOS environments.*


# Data Preprocessing Classes

In [ ]:
# Preprocessing step 1
def perform_column_pruning(data,th=75):
    data = data.replace(r'\s+', np.nan, regex=True)
    data[data == np.inf] = np.nan
    data = data.replace(r'^\s*$', np.nan, regex=True)

    na_sum_series = data.isna().mean()
    org_data = data.copy()

    NAN_data = pd.DataFrame({0: na_sum_series.index, 1: na_sum_series.values})
    dropped = []
    for i in range(len(NAN_data)):
          if NAN_data.iloc[i][1] >= (th / 100):
            dropped.append(NAN_data.iloc[i][0])
    data = data.drop(dropped, axis=1)
    for i in data:
        data[i] = pd.to_numeric(data[i])
    return data

In [ ]:
# Preprocessing step 2
def handle_missing_values(Idata):
    print('Processing Missing Values.', flush=True)

    data = Idata.drop(['SMILES'], axis=1)

    data.replace([np.inf, -np.inf, '', ' '], np.nan,inplace=True)
    print('Replaced spaces and inf.', flush=True)

    data.fillna(data.mean(),inplace=True)
    print('Averaged mean.', flush=True)

    data['SMILES'] = Idata['SMILES']
    del Idata

    # with open('preprocessed_feature_file.pkl', 'wb') as f:
    #     pickle.dump(data, f)
    # print('Saved preprocessed.', flush=True)

    print('Final count -', len(data), flush=True)
    return data

In [ ]:
# Preprocessing step 3
class Preprocess_Data:
    def VarianceRemoval(self,data_normal,thresh):
        self.data = data_normal
        selector = VarianceThreshold(thresh)
        selector.fit(data_normal)
        data_var_free=data_normal[data_normal.columns[selector.get_support(indices=True)]]
        return data_var_free

    def correlation_check(self,traindata,thresh): # drop columns above certain threshold
        self.data = traindata
        corr_matrix = traindata.corr()
        upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(np.bool))
        to_drop = [column for column in upper.columns if any(upper[column] >thresh)]
        trainset=traindata.drop(traindata[to_drop], axis=1)
        return trainset

    def Process_data(self,traindata):
        self.data = traindata
        traindata=self.VarianceRemoval(traindata,0.0)
        traindata=self.correlation_check(traindata,0.95)
        return traindata

In [ ]:
sign = Signaturizer('GLOBAL')
def Feature_Signaturizer(dat):

    """Generate Signaturizer features for a DataFrame containing a 'SMILES' column.

    Parameters
    ----------
    dat : pd.DataFrame
        Input table containing a 'SMILES' column.

    Returns
    -------
    pd.DataFrame
        A DataFrame with 'SMILES' and 3200 Signaturizer features named A1_0 ... E5_127.
    """

  if 'SMILES' not in dat.columns:
    raise ValueError("Input DataFrame must contain a 'SMILES' column.")

  print('Performing signaturizer...')

  smiles_list = dat['SMILES'].values

  # Generate signatures
  results = sign.predict(smiles_list)

  desc = ['A', 'B', 'C', 'D', 'E']

  feat_list = [
    f"{d}{i}_{j}"
    for d in desc          # A–E
    for i in range(1, 6)   # 1–5
    for j in range(128)   # 0–127
    ]

  # Build final DataFrame
  feature_df = pd.DataFrame(results.signature, columns=feat_list)
  final_df = pd.concat([dat[['SMILES']].reset_index(drop=True), feature_df], axis=1)

  print('✅ Signaturizer features generated successfully.')
  return handle_missing_values(final_df)

# Loading Data

In [ ]:
# -------------------------------
# Step 1: Load input compounds
# -------------------------------

input_smi = ["Cl[Cu]Cl", "CN(C(=O)NC)N=O", "COc1ccc(c2c1C(=O)CCC2)OC", "Oc1ccc2c(c1)C(=O)CCC2", "O=C1SCC(N1)C(=O)[O-].O=C1SCC(N1)C(=O)[O-].[Mg+2]"]

df = pd.DataFrame(input_smi, columns=['SMILES'])
df.head()

,SMILES
0,Cl[Cu]Cl
1,CN(C(=O)NC)N=O
2,COc1ccc(c2c1C(=O)CCC2)OC
3,Oc1ccc2c(c1)C(=O)CCC2
4,O=C1SCC(N1)C(=O)[O-].O=C1SCC(N1)C(=O)[O-].[Mg+2]


In [ ]:
# -------------------------------
# Step 2: Basic quality control
# -------------------------------

initial_count = df.shape[0]

df = df.dropna(subset=["SMILES"])
df = df.drop_duplicates(subset=["SMILES"])

print(f"Compounds retained: {df.shape[0]} / {initial_count}")

Compounds retained: 5 / 5


# Featurizing

In [ ]:
sign_df = Feature_Signaturizer(df)
sign_df.head()

Performing signaturizer...


Parsing SMILES: 5it [00:00, 1663.35it/s]
Generating signatures:   0%|          | 0/1 [00:00<?, ?it/s]

1/1 [==============================] - 0s 101ms/step


Generating signatures: 100%|██████████| 1/1 [00:00<00:00,  2.40it/s]

Signaturizer features generated successfully.
Processing Missing Values.
Replaced spaces and inf.


Averaged mean.
Final count - 5


,A1_0,A1_1,A1_2,A1_3,A1_4,A1_5,A1_6,A1_7,A1_8,A1_9,...,E5_119,E5_120,E5_121,E5_122,E5_123,E5_124,E5_125,E5_126,E5_127,SMILES
0,-0.095967,-0.085585,-0.095727,-0.072360,-0.095432,0.095816,0.095928,-0.091750,0.034593,-0.081889,...,-0.002504,-0.017272,0.022122,0.021757,0.010871,-0.016349,0.103875,0.170284,0.057716,Cl[Cu]Cl
1,-0.098109,0.096998,0.088180,0.061720,-0.055339,0.101028,0.100322,-0.100994,-0.072333,0.093840,...,-0.075915,0.013425,0.081162,0.055299,-0.054056,-0.114611,-0.049477,0.130626,0.007947,CN(C(=O)NC)N=O
2,-0.080750,-0.101839,-0.101258,0.048276,-0.011178,0.098917,-0.079727,0.100733,0.088806,-0.024737,...,-0.028702,-0.047627,0.106473,-0.008685,-0.115316,-0.089692,-0.103845,-0.026511,0.135247,COc1ccc(c2c1C(=O)CCC2)OC
3,-0.094732,-0.102159,-0.100932,0.018889,-0.053518,0.093207,-0.098040,0.102000,0.073988,0.035292,...,0.055529,-0.023970,0.002593,0.036640,0.012486,0.037370,-0.027829,0.009962,0.103282,Oc1ccc2c(c1)C(=O)CCC2
4,-0.102338,-0.102629,0.002752,-0.074643,0.092537,0.082024,-0.103412,-0.043952,0.073790,-0.084245,...,-0.007208,-0.105914,0.065345,0.025502,-0.085781,-0.124580,0.038651,0.064131,-0.007754,O=C1SCC(N1)C(=O)[O-].O=C1SCC(N1)C(=O)[O-].[Mg+2]




> 📌 *We can use the dataframe created above without further processing too.*



# Processing - Correlation and Variance Based

In [ ]:
data = sign_df.drop(["SMILES"], axis=1)
data = perform_column_pruning(data)
data = Preprocess_Data().Process_data(data)
data['SMILES'] = sign_df['SMILES']
data.head()

,A1_0,A1_1,A1_2,A1_3,A1_4,A1_5,A1_6,A1_7,A1_8,A1_9,...,B2_45,B2_47,B2_56,B2_68,B2_104,B2_122,B4_123,B5_27,B5_77,SMILES
0,-0.095967,-0.085585,-0.095727,-0.072360,-0.095432,0.095816,0.095928,-0.091750,0.034593,-0.081889,...,-0.006007,-0.088038,0.057849,0.000084,-0.162146,0.124928,-0.052945,-0.103066,-0.097069,Cl[Cu]Cl
1,-0.098109,0.096998,0.088180,0.061720,-0.055339,0.101028,0.100322,-0.100994,-0.072333,0.093840,...,-0.059600,-0.105155,0.006466,0.059996,-0.060591,-0.043605,0.084062,-0.107528,0.003968,CN(C(=O)NC)N=O
2,-0.080750,-0.101839,-0.101258,0.048276,-0.011178,0.098917,-0.079727,0.100733,0.088806,-0.024737,...,-0.149073,-0.104788,-0.109584,0.037586,-0.103561,-0.062720,0.026143,-0.131425,0.070779,COc1ccc(c2c1C(=O)CCC2)OC
3,-0.094732,-0.102159,-0.100932,0.018889,-0.053518,0.093207,-0.098040,0.102000,0.073988,0.035292,...,0.071542,-0.067482,0.027658,0.094086,-0.101855,0.044101,0.045566,-0.062251,-0.045441,Oc1ccc2c(c1)C(=O)CCC2
4,-0.102338,-0.102629,0.002752,-0.074643,0.092537,0.082024,-0.103412,-0.043952,0.073790,-0.084245,...,-0.028873,-0.084292,-0.013480,0.054491,-0.050085,0.059161,0.003565,-0.109066,-0.012078,O=C1SCC(N1)C(=O)[O-].O=C1SCC(N1)C(=O)[O-].[Mg+2]




> 📌 *This is the final dataframe that can be used for downstream analysis.*



## ✅ Summary & Next Steps

**What this notebook delivers**
- Fully standardized molecular inputs
- High-quality Signaturizer embeddings
- ML-ready feature matrices

**Intended downstream use**
- Supervised learning (classification/regression)
- Similarity-based screening
- Mechanism-aware clustering

**Extensible directions**
- Integration with contrastive learning
- Multi-omics fusion
- Pathway-specific modeling
